# marketgoblin — feature walkthrough

Runnable examples of **every feature, by provider**. Pairs with `docs/providers.md`.

| Provider | Auth | Catalog |
|---|---|---|
| `yahoo` | none | OHLCV, shares, dividends, metadata, classification |
| `tiingo` | API key | + splits, daily & quarterly fundamentals (paid) |

Cells using `yahoo` run with no setup. Cells marked **Tiingo (paid)** need a `TIINGO_API_KEY`.

## 0. Setup

Everything returns a Polars `LazyFrame` — nothing computes until `.collect()`.

In [ ]:
import logging
import tempfile
from pathlib import Path

import polars as pl

from marketgoblin import Dataset, MarketGoblin

logging.basicConfig(level=logging.INFO, format="%(levelname)s %(message)s")

START, END = "2024-01-01", "2024-03-31"

# A scratch dir so fetch() persists slices and load() can read them back.
_tmp = tempfile.TemporaryDirectory()
SAVE_PATH = Path(_tmp.name)
SAVE_PATH

### What does each provider support?

`supported_datasets` is the live source of truth — no need to guess from docs.

In [ ]:
for provider in ("yahoo", "tiingo"):
    try:
        g = MarketGoblin(provider=provider)
        datasets = sorted(d.value for d in g.supported_datasets)
    except Exception as exc:  # tiingo without a key, etc.
        datasets = f"<unavailable: {exc}>"
    print(f"{provider:8} -> {datasets}")

## 1. OHLCV — `Dataset.OHLCV` (yahoo, tiingo)

OHLCV is a **tidy stacked frame**: each trading day appears twice — once
`is_adjusted=True` (split/dividend adjusted) and once `False` (raw). Filter on
`is_adjusted` to pick a variant. There is no `adjusted=` toggle.

In [ ]:
goblin = MarketGoblin(provider="yahoo", save_path=SAVE_PATH)

ohlcv = goblin.fetch("AAPL", START, END, parse_dates=True).collect()
print(
    f"rows={ohlcv.height}  (adjusted={ohlcv.filter(pl.col('is_adjusted')).height}, "
    f"raw={ohlcv.filter(~pl.col('is_adjusted')).height})"
)
print(ohlcv.schema)
ohlcv.head(6)

In [ ]:
# Adjusted-only series, the common case for analysis.
adjusted = ohlcv.filter(pl.col("is_adjusted")).select(
    "date", "open", "high", "low", "close", "volume"
)
adjusted.head()

### Persistence & `load()`

With `save_path` set, `fetch()` writes monthly Parquet slices. `load()` reads
straight from disk — no network call.

In [ ]:
import json

from_disk = goblin.load("AAPL", START, END, parse_dates=True).collect()
print(f"loaded {from_disk.height} rows from disk")

# Inspect the JSON sidecar written next to the first slice.
sidecar = sorted(SAVE_PATH.rglob("ohlcv/**/*.json"))[0]
print(f"\nsidecar: {sidecar.name}")
json.loads(sidecar.read_text())

## 2. Shares outstanding — `Dataset.SHARES` (yahoo, tiingo)

Sparse, corporate-action cadence (not one row per day). yahoo dedups to the
last value per day; tiingo derives `shares = round(marketCap / close)`.

In [ ]:
shares = goblin.fetch("AAPL", START, END, dataset=Dataset.SHARES, parse_dates=True).collect()
print(shares.schema)
shares.head()

## 3. Dividends — `Dataset.DIVIDENDS` (yahoo, tiingo)

Event-driven cash dividends, filtered to `[start, end]`.

In [ ]:
dividends = goblin.fetch("AAPL", START, END, dataset=Dataset.DIVIDENDS, parse_dates=True).collect()
print(dividends.schema)
dividends

## 4. Batch fetch — `fetch_many()` (any dataset)

Concurrent, rate-limited. Failed symbols are logged and excluded — they never
crash the batch. Returns `{symbol: LazyFrame}`.

In [ ]:
results = goblin.fetch_many(
    ["AAPL", "MSFT", "GOOGL"],
    START,
    END,
    max_workers=4,
    requests_per_second=2.0,
)
for symbol, lf in results.items():
    print(f"{symbol}: {lf.collect().height} rows")

## 5. Ticker metadata — `fetch_metadata()` (yahoo, tiingo)

Returns a unified `TickerMetadata` dataclass (not a frame). `fast=True` skips
the slow/paid profile lookups. `load_metadata()` reads it back from disk.

In [ ]:
meta = goblin.fetch_metadata("AAPL")
print(f"{meta.symbol}: {meta.name}")
print(f"  exchange={meta.exchange}  currency={meta.currency}  sector={meta.sector}")
print(f"  market_cap={meta.market_cap}  trailing_pe={meta.trailing_pe}")

# Round-trips through disk because save_path is set.
reloaded = goblin.load_metadata("AAPL")
assert reloaded.symbol == meta.symbol
reloaded

## 6. Sector / industry classification — `fetch_classification()` (yahoo, tiingo)

Returns a `Classification` with nested `SectorProfile` / `IndustryProfile`.
Either may be `None` (ETFs, crypto, indices).

In [ ]:
clf = goblin.fetch_classification("AAPL")
if clf.sector:
    print(f"sector:   {clf.sector.name} (ETF {clf.sector.etf_symbol})")
if clf.industry:
    print(f"industry: {clf.industry.name}")
    print(f"  top companies: {clf.industry.top_companies[:5]}")

## 7. Sector → index mapping (provider-agnostic)

Shipped with the package; no provider or network needed.

In [ ]:
from marketgoblin import load_sector_indices

mapping = load_sector_indices("US")
type(mapping)

## 8. Tiingo (paid) — splits & fundamentals

These datasets are **Tiingo-only** and need a paid `TIINGO_API_KEY`
(env var or `api_key=`). The cell skips gracefully if no key is configured.

In [ ]:
import os

TIINGO_DATASETS = [
    Dataset.SPLITS,
    Dataset.FUNDAMENTALS_DAILY,
    Dataset.FUNDAMENTALS_STATEMENTS,
]

if not os.environ.get("TIINGO_API_KEY"):
    print("No TIINGO_API_KEY set - skipping paid Tiingo examples.")
else:
    tiingo = MarketGoblin(provider="tiingo", save_path=SAVE_PATH)
    for ds in TIINGO_DATASETS:
        df = tiingo.fetch("AAPL", "2023-01-01", END, dataset=ds, parse_dates=True).collect()
        print(f"\n=== {ds.value} ({df.height} rows) ===")
        print(df.schema)
        print(df.head(3))

## Cleanup

In [ ]:
_tmp.cleanup()